# NB_GLD_SRV_PRODUCT_OFFER

RECOMPUTE `lh_vv_gold.dbo.gld_srv_product_offer`. Một dòng mỗi `product_id`. Holiday và hotel cùng bảng. Variant nằm trong `variants_json`.


In [ ]:
p_exec_id = ""
p_run_id = ""
p_pl_name = ""
p_trg_fq = ""
p_pinned_json = ""
p_mode = "RUN"
p_force = "false"
p_history_limit = "1000"


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
spark.conf.set("spark.sql.legacy.timeParserPolicy", "LEGACY")

PARTNER = "partner"
PRICE_TIERS = ((500000, 1), (2000000, 2), (5000000, 3))
AXES = {
    "holiday:swticket": "variant_tickets", "holiday:use_date": "variant_dates",
    "holiday:hotel": "variant_hotels", "holiday:quantity": "variant_age_groups",
    "holiday:age_group": "variant_age_groups", "holiday:zone": "variant_room_types",
    "holiday:depature": "variant_departures",
}
AXIS_COLS = (
    "variant_tickets", "variant_dates", "variant_hotels", "variant_age_groups",
    "variant_room_types", "variant_departures",
)
MAP_T = "map<string,string>"
RAW_T = (
    "struct<shortDescription:string,translations:array<struct<languageCode:string,"
    "shortDescription:string>>,tags:array<struct<label:string,code:string>>>"
)
JSON_OPTS = {"ignoreNullFields": "false"}
HOTEL_JSON_OPTS = {
    "ignoreNullFields": "false", "timeZone": "UTC",
    "timestampFormat": "yyyy-MM-dd'T'HH:mm:ss'Z'",
}
BY_ID = (
    "(l, r) -> CASE WHEN l.variant_id < r.variant_id THEN -1 "
    "WHEN l.variant_id > r.variant_id THEN 1 ELSE 0 END"
)
INPUTS = {
    "sale": "slv_pn_product_sale", "room_offer": "slv_pn_hotel_room_offer",
    "rate_offer": "slv_pn_hotel_rate_offer", "products": "slv_pn_products",
    "product_tr": "slv_pn_product_translations", "variants": "slv_pn_product_variants",
    "variant_tr": "slv_pn_product_variant_translations", "attributes": "slv_pn_product_attributes",
    "attribute_values": "slv_pn_product_attribute_values",
    "attribute_value_tr": "slv_pn_product_attribute_value_translations",
    "posts": "slv_pn_product_posts", "relation": "slv_pn_product_business_service_relation",
    "room_types": "slv_pn_hotel_room_types", "rate_plans": "slv_pn_hotel_rate_plans",
    "destination": "slv_enr_poi_destination",
}
OUT_TYPES = {
    "product_id": "string", "name": "string", "lang": "string", "product_type": "string",
    "is_active": "boolean", "is_bookable": "boolean", "product_line": "string", "price_level": "int",
    "original_price": "double", "marketing_price": "double", "markup_percent": "double",
    "variant_count": "bigint", "active_variant_count": "bigint", "variants_json": "string",
    "variant_tickets": "string", "variant_dates": "string", "variant_hotels": "string",
    "variant_age_groups": "string", "variant_room_types": "string", "variant_departures": "string",
    "variant_statuses": "string", "product_attributes": "string", "business_service_id": "string",
    "source_id": "string", "product_post_id": "string", "poi_id": "string", "destination_id": "string",
}
TYPES = {
    "sale": {
        "product_id": "string", "product_post_id": "string", "original_price": "double",
        "marketing_price": "double", "markup_percent": "double", "is_bookable": "boolean",
    },
    "room_offer": {
        "room_type_id": "string", "status": "string", "view_type": "string",
        "available_rooms": "int", "max_occupancy": "int", "adult_capacity": "int",
        "child_capacity": "int", "is_bookable": "boolean",
    },
    "rate_offer": {
        "rate_plan_id": "string", "status": "string", "meal_plan": "int",
        "valid_from": "timestamp", "valid_to": "timestamp",
    },
    "products": {"id": "string", "product_line": "string"},
    "product_tr": {"product_id": "string", "locale": "string", "name": "string"},
    "variants": {
        "id": "string", "product_id": "string", "status": "string",
        "attributes": "string", "deleted": "boolean",
    },
    "variant_tr": {"variant_id": "string", "locale": "string", "name": "string"},
    "attributes": {"id": "string", "code": "string"},
    "attribute_values": {"id": "string", "attribute_id": "string", "code": "string"},
    "attribute_value_tr": {"attribute_id": "string", "locale": "string", "name": "string"},
    "posts": {"id": "string", "business_service_id": "string", "raw_sync_data": "string"},
    "relation": {"product_id": "string", "business_service_id": "string", "deleted": "boolean"},
    "room_types": {
        "id": "string", "name": "string", "description": "string", "tags": "string",
        "amenities": "string", "business_service_id": "string", "max_extra_bed": "int",
        "room_size": "int", "total_rooms": "int",
    },
    "rate_plans": {
        "id": "string", "business_service_id": "string", "name": "string", "kind": "string",
        "include_tax": "boolean", "include_service_charge": "boolean",
    },
    "destination": {"poi_id": "string", "destination_id": "string"},
}
NULL_STR = F.lit(None).cast("string")
NULL_DBL = F.lit(None).cast("double")
NULL_LNG = F.lit(None).cast("bigint")


def _require_types(df, expected, name):
    """Chặn khi build thiếu cột hoặc lệch kiểu. Không cast.

    Input : df — DataFrame nguồn; expected — {cột: simpleString}; name — tên bảng
    Output: None
    Ví dụ : _require_types(sale, TYPES["sale"], "slv_pn_product_sale")
    """
    got = {field.name: field.dataType.simpleString() for field in df.schema.fields}
    missing = [col for col in expected if col not in got]
    bad = [f"{col}: {got[col]}" for col in expected if col in got and got[col] != expected[col]]
    if missing or bad:
        raise ConfigError(f"{name}: thiếu {missing}; lệch kiểu {bad}")


def _require_unique(df, keys, name):
    """Trùng khoá tra cứu thì dừng trước khi ghi.

    Input : df — DataFrame đã lọc; keys — cột khoá; name — tên kiểm tra
    Output: None
    Ví dụ : _require_unique(products, ("id",), "slv_pn_products")
    """
    n = df.groupBy(*keys).count().where("count > 1").count()
    if n:
        raise QualityGateError(f"gld_srv_product_offer: {name} trùng khoá {keys}: {n}")


def _price_level():
    """Bậc giá holiday. NULL -> 0. Ngoài ba mốc -> 4. Không giải mã."""
    p = F.coalesce(F.col("marketing_price"), F.col("original_price"))
    expr = F.when(p.isNull(), F.lit(0))
    for limit, level in PRICE_TIERS:
        expr = expr.when(p <= limit, F.lit(level))
    return expr.otherwise(F.lit(4)).cast("int")


def _sorted_json(name, opts):
    """variants_json không NULL. Sắp theo variant_id. opts giữ timestamp hotel."""
    col = F.col(name)
    return F.when(col.isNull(), F.lit("[]")).otherwise(
        F.to_json(F.expr(f"array_sort({name}, {BY_ID})"), opts)
    )


def _read(ctx, role):
    df = ctx.input(role)
    _require_types(df, TYPES[role], INPUTS[role])
    return df


def _spec():
    return {
        "trg_tbl": "gld_srv_product_offer",
        "inputs": INPUTS,
        "pk": ("product_id",),
        "required_inputs": ("sale", "room_offer"),
        "hash_exclude": ("is_active",),
        "soft_delete_set": {"is_active": "false"},
        "write_mode": "MERGE",
    }


def _holiday(ctx):
    """Nhánh holiday: một dòng sale còn sống, bỏ product nhiều business_service_id."""
    sale = _read(ctx, "sale")
    products = _read(ctx, "products")
    product_tr = _read(ctx, "product_tr")
    variants = _read(ctx, "variants")
    variant_tr = _read(ctx, "variant_tr")
    attributes = _read(ctx, "attributes")
    attribute_values = _read(ctx, "attribute_values")
    attribute_value_tr = _read(ctx, "attribute_value_tr")
    posts = _read(ctx, "posts")
    relation = _read(ctx, "relation")
    _require_unique(products, ("id",), "slv_pn_products")
    _require_unique(posts, ("id",), "slv_pn_product_posts")
    tr = product_tr.where(F.col("locale").isin("en", "vi"))
    _require_unique(tr, ("product_id", "locale"), "slv_pn_product_translations")
    vtr = variant_tr.where(F.col("locale") == F.lit("vi"))
    _require_unique(vtr, ("variant_id",), "slv_pn_product_variant_translations")
    live = variants.where(F.col("deleted") == F.lit(False))
    _require_unique(live, ("id",), "slv_pn_product_variants")

    rel_agg = (
        relation.where(F.col("deleted") == F.lit(False))
        .groupBy("product_id")
        .agg(
            F.count(F.lit(1)).alias("_rel_rows"),
            F.countDistinct("business_service_id").alias("_rel_n"),
            F.max("business_service_id").alias("_rel_bsid"),
        )
    )
    posts_s = posts.select(
        F.col("id").alias("_post_id"),
        F.col("business_service_id").alias("_post_bsid"),
        "raw_sync_data",
    )
    base = sale.join(rel_agg, "product_id", "left").join(
        posts_s, sale["product_post_id"] == posts_s["_post_id"], "left",
    )
    dropped_multi_bsid = base.where(F.col("_rel_n") > 1).count()
    base = base.where(F.coalesce(F.col("_rel_n"), F.lit(0)) <= 1)
    relation_null_bsid_only = base.where(
        F.col("_rel_rows").isNotNull() & (F.col("_rel_n") == 0)
    ).count()
    product_missing = base.join(products.select("id"), base["product_id"] == products["id"], "left_anti").count()
    base = base.withColumn(
        "business_service_id",
        F.when(F.col("_rel_rows").isNull(), F.col("_post_bsid")).when(F.col("_rel_n") == 1, F.col("_rel_bsid")),
    )

    named = live.join(
        vtr.select(F.col("variant_id").alias("id"), F.col("name").alias("_vname")),
        "id", "left",
    )
    named = named.withColumn(
        "_attrs",
        F.when(F.col("attributes").isNull(), F.from_json(F.lit("{}"), MAP_T)).otherwise(
            F.from_json(F.col("attributes"), MAP_T)
        ),
    )
    named = named.withColumn(
        "_unparsed", F.col("attributes").isNotNull() & F.col("_attrs").isNull(),
    )
    variant_attributes_unparsed = named.where(F.col("_unparsed")).count()
    grouped = named.groupBy("product_id").agg(
        F.collect_list(F.struct(
            F.col("id").alias("variant_id"),
            F.col("_vname").alias("name"),
            F.col("status").alias("status"),
            F.col("_attrs").alias("attributes"),
        )).alias("_v"),
        F.count(F.lit(1)).alias("variant_count"),
        F.sum(F.when(F.col("status") == F.lit("active"), 1).otherwise(0)).cast("bigint").alias("active_variant_count"),
        F.array_sort(F.collect_set("status")).alias("_st"),
    )
    exploded = named.select(
        "product_id", F.explode(F.map_entries(F.col("_attrs"))).alias("_e"),
    ).select(
        "product_id",
        F.col("_e.key").alias("attr_code"),
        F.col("_e.value").alias("value_code"),
    )
    axis_map = F.create_map(*[item for pair in AXES.items() for item in (F.lit(pair[0]), F.lit(pair[1]))])
    labels = (
        exploded
        .join(attributes.select(F.col("id").alias("_aid"), F.col("code").alias("attr_code")), "attr_code", "left")
        .join(
            attribute_values.select(
                F.col("id").alias("_vid"), F.col("attribute_id").alias("_aid"), F.col("code").alias("value_code"),
            ),
            ["_aid", "value_code"], "left",
        )
        .join(
            attribute_value_tr.where(F.col("locale") == F.lit("vi")).select(
                F.col("attribute_id").alias("_vid"), F.col("name").alias("_label"),
            ),
            "_vid", "left",
        )
        .withColumn("_axis", F.element_at(axis_map, F.col("attr_code")))
    )
    axis_agg = [F.array_sort(F.collect_set(F.when(F.col("_axis") == F.lit(col), F.col("_label")))).alias(col) for col in AXIS_COLS]
    axes = labels.groupBy("product_id").agg(*axis_agg)
    en = tr.where(F.col("locale") == F.lit("en")).select(
        "product_id", F.col("name").alias("_en"),
    )
    vi = tr.where(F.col("locale") == F.lit("vi")).select(
        "product_id", F.col("name").alias("_vi"),
    )
    frame = (
        base
        .join(grouped, "product_id", "left")
        .join(axes, "product_id", "left")
        .join(en, "product_id", "left")
        .join(vi, "product_id", "left")
        .join(products.select(F.col("id").alias("product_id"), "product_line"), "product_id", "left")
    )
    frame = frame.withColumn("r", F.from_json(F.col("raw_sync_data"), RAW_T))
    en_tr = F.filter(F.col("r.translations"), lambda row: row["languageCode"].startswith("en"))
    vi_tr = F.filter(F.col("r.translations"), lambda row: row["languageCode"].startswith("vi"))
    short = F.coalesce(
        F.when(F.size(en_tr) > 0, en_tr.getItem(0)["shortDescription"]),
        F.when(F.size(vi_tr) > 0, vi_tr.getItem(0)["shortDescription"]),
        F.col("r.shortDescription"),
    )
    tag_values = F.transform(F.col("r.tags"), lambda row: F.coalesce(row["label"], row["code"]))
    tags = F.when(F.size(tag_values) > 0, tag_values)
    frame = frame.withColumn(
        "product_attributes",
        F.to_json(F.struct(
            short.alias("shortDescription"),
            tags.alias("tags"),
            _price_level().alias("priceLevel"),
            F.lit(True).alias("isActive"),
        ), JSON_OPTS),
    )
    rows = frame.select(
        "product_id",
        F.coalesce(F.col("_en"), F.col("_vi"), F.col("product_id")).alias("name"),
        F.lit("en").alias("lang"),
        F.lit("other").alias("product_type"),
        F.lit(True).alias("is_active"),
        "is_bookable",
        "product_line",
        _price_level().alias("price_level"),
        "original_price", "marketing_price", "markup_percent",
        "variant_count", "active_variant_count",
        _sorted_json("_v", JSON_OPTS).alias("variants_json"),
        *[
            F.when(F.size(F.col(col)) > 0, F.to_json(F.col(col), JSON_OPTS)).alias(col)
            for col in AXIS_COLS
        ],
        F.when(F.size(F.col("_st")) > 0, F.to_json(F.col("_st"), JSON_OPTS)).alias("variant_statuses"),
        "product_attributes",
        "business_service_id",
        F.col("business_service_id").cast("string").alias("source_id"),
        "product_post_id",
        uuid_md5(F.col("business_service_id"), F.lit(PARTNER)).alias("poi_id"),
        NULL_STR.alias("destination_id"),
    )
    return rows, {
        "dropped_multi_bsid": dropped_multi_bsid,
        "variant_attributes_unparsed": variant_attributes_unparsed,
        "product_missing": product_missing,
        "relation_null_bsid_only": relation_null_bsid_only,
    }


def _hotel(ctx):
    """Nhánh hotel: một dòng room offer còn sống. Rate plan nằm trong variants_json."""
    room_offer = _read(ctx, "room_offer")
    rate_offer = _read(ctx, "rate_offer")
    room_types = _read(ctx, "room_types")
    rate_plans = _read(ctx, "rate_plans")
    _require_unique(room_types, ("id",), "slv_pn_hotel_room_types")
    _require_unique(rate_plans, ("id",), "slv_pn_hotel_rate_plans")
    types = room_types.select(
        F.col("id").alias("room_type_id"),
        F.col("name").alias("_rname"),
        "description", "tags", "amenities", "max_extra_bed", "room_size", "total_rooms",
        "business_service_id",
    )
    plans_in = rate_plans.select(
        F.col("id").alias("rate_plan_id"),
        "business_service_id",
        F.col("name").alias("_pname"),
        "kind", "include_tax", "include_service_charge",
        F.lit(1).alias("_plan_hit"),
    )
    joined = rate_offer.join(plans_in, "rate_plan_id", "left")
    rate_plan_missing = joined.where(F.col("_plan_hit").isNull()).count()
    kept = joined.where(F.col("business_service_id").isNotNull())
    plans = kept.groupBy("business_service_id").agg(
        F.collect_list(F.struct(
            F.col("rate_plan_id").alias("variant_id"),
            F.coalesce(F.col("_pname"), F.col("rate_plan_id")).alias("name"),
            F.col("status").alias("status"),
            F.struct(
                "meal_plan", "kind", "valid_from", "valid_to", "include_tax", "include_service_charge",
            ).alias("attributes"),
        )).alias("_v"),
        F.count(F.lit(1)).alias("_n"),
        F.sum(F.when(F.col("status") == F.lit("Active"), 1).otherwise(0)).cast("bigint").alias("_a"),
    )
    rooms = room_offer.join(types, "room_type_id", "left").join(plans, "business_service_id", "left")
    room_type_missing = room_offer.join(types.select("room_type_id"), "room_type_id", "left_anti").count()
    rows = rooms.select(
        F.col("room_type_id").alias("product_id"),
        F.coalesce(F.col("_rname"), F.col("room_type_id")).alias("name"),
        F.lit("en").alias("lang"),
        F.lit("hotel").alias("product_type"),
        F.lit(True).alias("is_active"),
        "is_bookable",
        F.lit("accommodation").alias("product_line"),
        F.lit(0).cast("int").alias("price_level"),
        NULL_DBL.alias("original_price"),
        NULL_DBL.alias("marketing_price"),
        NULL_DBL.alias("markup_percent"),
        F.coalesce(F.col("_n"), F.lit(0)).cast("bigint").alias("variant_count"),
        F.coalesce(F.col("_a"), F.lit(0)).cast("bigint").alias("active_variant_count"),
        _sorted_json("_v", HOTEL_JSON_OPTS).alias("variants_json"),
        *[NULL_STR.alias(col) for col in AXIS_COLS],
        NULL_STR.alias("variant_statuses"),
        F.to_json(F.struct(
            F.col("description").alias("shortDescription"),
            F.col("tags").alias("tags"),
            NULL_LNG.cast("int").alias("priceLevel"),
            F.lit(True).alias("isActive"),
            F.col("max_occupancy").alias("maxOccupancy"),
            F.col("adult_capacity").alias("adultCapacity"),
            F.col("child_capacity").alias("childCapacity"),
            F.col("max_extra_bed").alias("maxExtraBed"),
            F.col("room_size").alias("roomSize"),
            F.col("view_type").alias("viewType"),
            F.col("amenities").alias("amenities"),
            F.col("available_rooms").alias("availableRooms"),
            F.col("total_rooms").alias("totalRooms"),
        ), JSON_OPTS).alias("product_attributes"),
        "business_service_id",
        F.col("business_service_id").cast("string").alias("source_id"),
        NULL_STR.alias("product_post_id"),
        uuid_md5(F.col("business_service_id"), F.lit(PARTNER)).alias("poi_id"),
        NULL_STR.alias("destination_id"),
    )
    return rows, {"room_type_missing": room_type_missing, "rate_plan_missing": rate_plan_missing}


def build(ctx):
    """Union hai nhánh, gắn destination, chặn nhánh rỗng trước MERGE."""
    hol, hol_stats = _holiday(ctx)
    hot, hot_stats = _hotel(ctx)
    rows = hol.select(*OUT_TYPES).unionByName(hot.select(*OUT_TYPES))
    _require_unique(rows, ("product_id",), "union product_id")
    des = ctx.input("destination", live_only=False)
    _require_types(des, TYPES["destination"], "slv_enr_poi_destination")
    keys = rows.where(F.col("poi_id").isNotNull()).select("poi_id").distinct()
    matched = des.select("poi_id", "destination_id").join(keys, "poi_id", "left_semi")
    _require_unique(matched, ("poi_id",), "slv_enr_poi_destination")
    out = rows.drop("destination_id").join(matched, "poi_id", "left").select(*OUT_TYPES)
    _require_types(out, OUT_TYPES, "gld_srv_product_offer")
    out = out.persist(StorageLevel.MEMORY_AND_DISK)
    ctx.cached.append(out)
    other = F.lit("other")
    hotel = F.lit("hotel")
    summary = out.agg(
        F.sum(F.when(F.col("product_type") == other, 1).otherwise(0)).alias("holiday_rows"),
        F.sum(F.when(F.col("product_type") == hotel, 1).otherwise(0)).alias("hotel_rows"),
        F.sum(F.when(F.col("business_service_id").isNull(), 1).otherwise(0)).alias("null_business_service_id"),
        F.sum(F.when(
            F.col("poi_id").isNotNull() & F.col("destination_id").isNull(), 1,
        ).otherwise(0)).alias("no_destination"),
        F.sum(F.when(
            (F.col("product_type") == other) & (F.col("variants_json") == F.lit("[]")), 1,
        ).otherwise(0)).alias("empty_variants_holiday"),
        F.sum(F.when(
            (F.col("product_type") == hotel) & (F.col("variants_json") == F.lit("[]")), 1,
        ).otherwise(0)).alias("empty_variants_hotel"),
        F.sum(F.when(
            (F.col("product_type") == other)
            & F.col("product_post_id").isNotNull()
            & F.col("original_price").isNull(),
            1,
        ).otherwise(0)).alias("null_price_with_post"),
    ).first()
    metrics = {name: int(summary[name] or 0) for name in summary.asDict()}
    ctx.metrics.update({**hol_stats, **metrics})
    warns = {**hol_stats, **hot_stats}
    for name in ("product_missing", "room_type_missing", "rate_plan_missing"):
        ctx.warn(name, warns[name])
    if not metrics["holiday_rows"] or not metrics["hotel_rows"]:
        raise QualityGateError(
            f"gld_srv_product_offer: nhánh rỗng holiday={metrics['holiday_rows']} hotel={metrics['hotel_rows']}"
        )
    return out


SPEC = NodeSpec(
    trg_tbl="gld_srv_product_offer",
    inputs=INPUTS,
    pk=("product_id",),
    build=build,
    required_inputs=("sale", "room_offer"),
    hash_exclude=("is_active",),
    soft_delete_set={"is_active": "false"},
    write_mode="MERGE",
)


In [ ]:
RES = run_node(SPEC, p_exec_id, p_run_id, p_pinned_json, p_mode, p_force, p_pl_name, p_trg_fq, p_history_limit)


In [ ]:
notebookutils.notebook.exit(finish_node(RES))
